# GRIB2 to JMV Data Converter

Made by Kai Hatton

Welcome to this interactive Google Colab notebook designed to simplify the conversion of GRIB2 meteorological data into the JMV file format. This tool is especially useful for users in maritime and meteorological domains who rely on JMV files for specialized applications, but often receive primary weather data in the GRIB2 format.

**Purpose:**

This notebook provides a **no-code-required** experience for transforming GRIB2 files (e.g., from GFS, ECMWF) into JMV files directly within your browser using Google Colab.

**How to Use:**

Simply run the cells in order from top to bottom by clicking the 'Play' button (▶) next to each cell. Follow the instructions in the markdown cells to select your data source and download your converted files.

## 1. Initial Setup

Before we can start processing GRIB2 files, we need to install a few essential libraries. These libraries handle GRIB2 parsing, data manipulation, and file operations. The cell below will install everything needed in your Colab environment.

In [2]:
# Install Python packages
!pip install --quiet ecmwf-opendata cfgrib xarray boto3 pandas requests

# Install system libraries required by cfgrib (for GRIB2 decoding)
!apt-get install --quiet -y libeccodes-dev

print("Installation complete. You might see some warning messages, but as long as no fatal errors occurred, you can proceed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 73.0 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libeccodes-data libeccodes0
The following NEW packages will be installed:
  libeccodes-data libeccodes-dev libeccodes0
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,076 kB of archives.
After this operation

In [3]:
# Import all necessary libraries
import numpy as np
import os
import xarray as xr
import cfgrib
import shutil
from datetime import datetime, timedelta
import zipfile
import pandas as pd
from google.colab import files
import requests
from io import BytesIO
import csv # Added for reading CSV parameter files

# Function to load parameter specifications from a CSV file.
# This function returns raw data that can be used to augment PARAM_SPECS later.
def load_param_specs_from_csv(filepath: str) -> list:
    """
    Loads parameter specifications from a CSV file.
    Assumes CSV format: JMV_KEY,GRIB_KEY,SCALE,OFFSET,UNITS (no header).
    Returns a list of dictionaries, each representing a parameter.
    """
    additional_params_raw = []
    try:
        with open(filepath, 'r') as f:
            reader = csv.reader(f)
            for line_num, row in enumerate(reader):
                if not row or row[0].startswith('#'): # Skip empty lines or comments
                    continue
                try:
                    if len(row) == 5:
                        jmv_key = row[0].strip()
                        grib_key = row[1].strip()
                        scale = float(row[2].strip())
                        offset = float(row[3].strip())
                        units = row[4].strip()
                        additional_params_raw.append({
                            'key': jmv_key,
                            'grib_key': grib_key,
                            'scale': scale,
                            'offset': offset,
                            'units': units
                        })
                    else:
                        print(f"Warning: Skipping malformed line {line_num+1} in {filepath}: {row}. Expected 5 columns.")
                except ValueError as ve:
                    print(f"Warning: Skipping line {line_num+1} in {filepath} due to data type error: {row} - {ve}")
        print(f"Successfully read raw parameters from {filepath}.")
    except FileNotFoundError:
        print(f"Error: Parameter file not found at {filepath}")
    except Exception as e:
        print(f"An error occurred while loading parameters from {filepath}: {e}")
    return additional_params_raw

print("Libraries imported successfully.")
print("A function `load_param_specs_from_csv` has been defined to read additional parameters from '/content/GRIB2JMVALPHA.txt'.")

Libraries imported successfully.
A function `load_param_specs_from_csv` has been defined to read additional parameters from '/content/GRIB2JMVALPHA.txt'.


## 2. Core Conversion Logic (Backend)

The cells below contain the core functions that perform the GRIB2 to JMV conversion. You **do not need to modify** these cells. Simply run them to define the necessary logic for the notebook to work.

In [4]:
from __future__ import annotations

import os
from dataclasses import dataclass, field

import numpy as np

# xarray / cfgrib are imported lazily inside the functions that need them so
# this module can be imported (and the JMV writer unit-tested) on a machine
# without eccodes installed.


# --------------------------------------------------------------------------- #
# Parameter specifications (FNMOC conventions -- preserved from the original)  #
# --------------------------------------------------------------------------- #
@dataclass
class ParamSpec:
    """One output product. ``grib_tag`` is the eccodes shortName used to pull
    the variable out of the GRIB2 file."""
    grib_tag: str
    product_title: str
    grib_parameter_id: int
    units_code: int
    sort_level: int = 100
    process_id: int = 110
    data_type_code: int = 7
    grib_modifier_type: int = 1
    grib_modifier: float = 1.0
    data_multiplier: float = 0.1
    kelvin_to_celsius: bool = False
    missing_value: float = 0.0


PARAM_SPECS: dict[str, ParamSpec] = {
    "tmp":  ParamSpec("2t",   "2 METRE TEMPERATURE",        11, 11,  sort_level=100,
                      data_multiplier=0.1, kelvin_to_celsius=True),
    "gust": ParamSpec("gust", "WIND GUST",                 180, 180, sort_level=112,
                      data_multiplier=0.1),
    "msl":  ParamSpec("msl",  "MEAN SEA LEVEL PRESSURE",     1, 1,   sort_level=100,
                      data_multiplier=0.1),
    "10u":  ParamSpec("10u",  "10 METRE U WIND COMPONENT",  33, 33,  sort_level=100,
                      data_multiplier=0.1),
    "10v":  ParamSpec("10v",  "10 METRE V WIND COMPONENT",  34, 34,  sort_level=100,
                      data_multiplier=0.1),
    "mwd":  ParamSpec("mwd",  "MEAN WAVE DIRECTION",       107, 107, sort_level=100,
                      data_multiplier=0.1),
    "mwp":  ParamSpec("mwp",  "MEAN WAVE PERIOD",          108, 108, sort_level=100,
                      data_multiplier=0.1),
    "swh":  ParamSpec("swh",  "SIGNIFICANT WAVE HEIGHT",   100, 100, sort_level=100,
                      data_multiplier=0.01),
}


# --------------------------------------------------------------------------- #
# JMV header + writer (no eccodes required)                                   #
# --------------------------------------------------------------------------- #
HEADER_TERMINATOR = "END_OF_HEADER"


def _encode_header(header: dict) -> bytes:
    lines = [f"{k} = {v}" for k, v in header.items()]
    return ("\n".join(lines) + f"\n{HEADER_TERMINATOR}\n").encode("ascii")


def write_jmv(file_path: str, header: dict, data: np.ndarray,
              missing_value: float = 0.0) -> str:
    """Write a 2-D grid + ASCII header to a JMV file.

    The header's ``DATA_OFFSET`` is filled in with the real byte offset of the
    grid (two-pass: the placeholder and the final value are both 5-char
    zero-padded, so the header length is stable).
    """
    data = np.asarray(data, dtype=float)
    if data.ndim == 3:
        data = data.squeeze()
    if data.ndim != 2:
        raise ValueError(f"Data must be 2-D after squeeze, got shape {data.shape}")

    grib_modifier = float(header.get("GRIB_MODIFIER", 1.0)) or 1.0
    data_multiplier = float(header.get("DATA_MULTIPLIER", 1.0)) or 1.0

    # Replace NaNs before integer cast (prevents INT_MIN overflow artifacts).
    clean = np.nan_to_num(data, nan=float(missing_value))
    scaled = np.round((clean * grib_modifier) / data_multiplier)

    # Explicit little-endian int32 so output is byte-for-byte reproducible
    # regardless of host architecture.
    grid_bytes = scaled.astype("<i4").tobytes()

    header["NUMBER_OF_RECORDS"] = str(data.size)
    header["DATA_OFFSET"] = "00000"                      # placeholder, 5 chars
    header["DATA_OFFSET"] = str(len(_encode_header(header))).zfill(5)
    final_header = _encode_header(header)

    with open(file_path, "wb") as f:
        f.write(final_header)
        f.write(grid_bytes)
    return file_path


def build_header(spec: ParamSpec, da, grib_file_path: str) -> dict:
    """Assemble the JMV ASCII header for one parameter from its DataArray."""
    native = da.values.astype(float)
    if native.ndim > 2:
        native = native.squeeze()
    if spec.kelvin_to_celsius and np.nanmean(native) > 200:
        native = native - 273.15

    dtg = da.time.dt.strftime("%Y%m%d%H%M").item()
    tau = int(da.step.dt.total_seconds().item() // 3600) if "step" in da.coords else 0
    lat_n = int(da.sizes["latitude"])
    lon_n = int(da.sizes["longitude"])
    lat_spacing = abs(float(da.latitude[1] - da.latitude[0])) if lat_n > 1 else 0.25
    lon_spacing = abs(float(da.longitude[1] - da.longitude[0])) if lon_n > 1 else 0.25

    return {
        "VERSION": "1.0",
        "DATA_OFFSET": "0",
        "PLATFORM": "PC",
        "PRODUCT_TITLE": spec.product_title,
        "DATA_BASE_TITLE": spec.product_title,
        "CENTER_ID": "58",
        "PROCESS_ID": str(spec.process_id),
        "PRODUCT_TYPE": "GD",
        "UNKNOWN_PRODUCT_CODE": "0",
        "SORT_LEVEL": str(spec.sort_level),
        "DATA_TYPE_CODE": str(spec.data_type_code),
        "LABEL_CENTER_VALUE": "1",
        "GRIB_FILE_NAME": os.path.basename(grib_file_path),
        "GRIB_PARAMETER_ID": str(spec.grib_parameter_id),
        "GRIB_UNITS_CODE": "0",
        "UNITS_CODE": str(spec.units_code),
        "DATE_TIME_GROUP": dtg,
        "REQUESTED_TAU": str(tau),
        "DELIVERED_TAU": str(tau),
        "LEVEL_INDICATOR": "1",
        "LEVEL": "0.000000",
        "STANDARD_HEIGHT": "0.000000",
        "MB_LEVEL": "0.000000",
        "MODEL": "ECMWF",
        "BASE_LONGITUDE": f"{float(da.longitude.min()):.6f}",
        "BOTTOM_LATITUDE": f"{float(da.latitude.min()):.6f}",
        "PROJECTION": "1",
        "POLE_ON_SCREEN": "0",
        "LAT_POINTS": str(lat_n),
        "LON_POINTS": str(lon_n),
        "LABEL_LENGTH_CODE": "0",
        "HIGH_LOW_FLAG": "0",
        "TITLE_TYPE": "0",
        "DATA_MAX_VALUE": f"{np.nanmax(native):.1f}",
        "DATA_MIN_VALUE": f"{np.nanmin(native):.1f}",
        "ORIG_DATA_MAX_VALUE": f"{np.nanmax(native):.6f}",
        "ORIG_DATA_MIN_VALUE": f"{np.nanmin(native):.6f}",
        "CONTOUR_ORIGIN": "3",
        "CONTOUR_INTERVAL": "3",
        "CONTOUR_INTERVAL_COMPUTED": "NO",
        "CONTOUR_HIGH": "9999",
        "GRIB_MODIFIER_TYPE": str(spec.grib_modifier_type),
        "GRIB_MODIFIER": f"{spec.grib_modifier:.6f}",
        "DATA_MULTIPLIER": f"{spec.data_multiplier:.6f}",
        "LAND_SEA_FLAG": "1",
        "LATITUDE_GRID_SPACING": f"{lat_spacing:.6f}",
        "LONGITUDE_GRID_SPACING": f"{lon_spacing:.6f}",
        "PARTS_PER_RECORD": "1",
        "BYTES_PER_RECORD": "4",
        "RECORD_TYPE_PART1": "INTEGER",
        "BYTES_PER_POINT_PART1": "4",
    }, native


# --------------------------------------------------------------------------- #
# GRIB reading (requires eccodes / cfgrib)                                     #
# --------------------------------------------------------------------------- #
def _open_param(grib_file_path: str, short_name: str):
    """Open a single variable from a GRIB2 file by shortName, or return None."""
    import xarray as xr
    try:
        ds = xr.open_dataset(
            grib_file_path,
            engine="cfgrib",
            backend_kwargs={
                "filter_by_keys": {"shortName": short_name},
                "indexpath": "",
            },
        )
    except Exception:
        return None
    var_names = list(ds.data_vars)
    if not var_names:
        ds.close()
        return None
    return ds, ds[var_names[0]]


def detect_parameters(filepath):
    """Detect GRIB2 parameters. Return (params_list, error_msg_or_None)."""
    params = []
    try:
        with cfgrib.open_datasets(filepath) as dss:
            for ds in dss:
                params.extend(list(ds.data_vars.keys()))
    except Exception as e:
        error_msg = str(e)
        if "Edition not supported" in error_msg:
            return [], "File is not GRIB2 format (Edition 2). Check the file type."
        elif "corrupted" in error_msg.lower():
            return [], "File appears corrupted. Try re-downloading it."
        else:
            return [], f"Cannot read file: {error_msg}"
    return params, None


def _jmv_filename(spec: ParamSpec, da, lon_n: int, lat_n: int) -> str:
    dtg = da.time.dt.strftime("%Y%m%d%H%M").item()
    tau = int(da.step.dt.total_seconds().item() // 3600) if "step" in da.coords else 0
    safe = spec.product_title.replace(" ", "_")
    return f"{safe}^GD^NCEP_GFS_{lon_n}X{lat_n}^{dtg}^0^{tau}.JMV"


def convert_grib_to_jmv(grib_file_path: str, output_dir: str,
                        param_specs: dict[str, ParamSpec] | None = None,
                        progress=None) -> list[str]:
    """Convert every parameter found in ``grib_file_path`` to a JMV file.

    Returns the list of written file paths. ``progress`` is an optional
    callable ``(stage: str, done: int, total: int)`` for UI updates.
    """
    specs = param_specs or PARAM_SPECS
    os.makedirs(output_dir, exist_ok=True)
    written: list[str] = []
    items = list(specs.items())

    for i, (key, spec) in enumerate(items):
        if progress:
            progress(spec.product_title, i, len(items))
        opened = _open_param(grib_file_path, spec.grib_tag)
        if opened is None:
            continue
        ds, da = opened
        try:
            header, native = build_header(spec, da, grib_file_path)
            lat_n, lon_n = int(da.sizes["latitude"]), int(da.sizes["longitude"])
            fname = _jmv_filename(spec, da, lon_n, lat_n)
            out_path = os.path.join(output_dir, fname)
            write_jmv(out_path, header, native, missing_value=spec.missing_value)
            written.append(out_path)
        except Exception as exc:  # one bad parameter shouldn't kill the batch
            print(f"  [!] {spec.grib_tag}: {exc}")
        finally:
            ds.close()

    if progress:
        progress("done", len(items), len(items))
    return written


def preview_grid(grib_file_path: str, param_key: str, max_dim: int = 180) -> dict:
    """Downsample one parameter to a small 2-D array for the UI heatmap.

    Returns ``{values, nx, ny, vmin, vmax, title}`` (values row-major, north-up).
    """
    spec = PARAM_SPECS[param_key]
    opened = _open_param(grib_file_path, spec.grib_tag)
    if opened is None:
        raise ValueError(f"{spec.product_title} not present in file")
    ds, da = opened
    try:
        arr = da.values.astype(float)
        if arr.ndim > 2:
            arr = arr.squeeze()
        if spec.kelvin_to_celsius and np.nanmean(arr) > 200:
            arr = arr - 273.15
        # north-up: GRIB latitudes usually descend (90 -> -90); flip if ascending
        if float(da.latitude[0]) < float(da.latitude[-1]):
            arr = arr[::-1, :]
        ny, nx = arr.shape
        sy = max(1, ny // max_dim)
        sx = max(1, nx // max_dim)
        small = arr[::sy, ::sx]
        finite = small[np.isfinite(small)]
        vmin = float(np.percentile(finite, 2)) if finite.size else 0.0
        vmax = float(np.percentile(finite, 98)) if finite.size else 1.0
        return {
            "values": np.nan_to_num(small, nan=vmin).round(3).flatten().tolist(),
            "nx": int(small.shape[1]),
            "ny": int(small.shape[0]),
            "vmin": vmin,
            "vmax": vmax,
            "title": spec.product_title,
        }
    finally:
        ds.close()

## 3. Choose Your Data Source

Now it's time to get your GRIB2 data! You have two main options:

*   **Method A: Download GRIB2 Data Automatically** - This option allows you to specify a data source (like GFS) and a date range, and the notebook will attempt to download the corresponding GRIB2 files for you.
*   **Method B: Upload Your Own GRIB2 Files** - If you already have GRIB2 files on your computer, you can upload them in a ZIP archive, and the notebook will process them.

**Please choose and run ONLY ONE of the following methods (Method A or Method B).**

### Method A: Download GRIB2 Data Automatically

This section allows you to download GRIB2 data directly from public data sources (e.g., NOAA GFS). Use the interactive form below to select your desired options. Once you've made your selections, run the next cell to start the download and conversion process.

In [14]:
from ecmwf.opendata import Client

#@markdown ### ECMWF Data Download Options
#@markdown Configure your ECMWF data request here.

ecmwf_model_run_date_str = '2026-07-09' #@param {type:"date"}
ecmwf_model_cycle = '06' #@param ["00", "06", "12", "18"]
ecmwf_max_forecast_hour = 240 #@param {type:"slider", min:0, max:240, step:24}
ecmwf_finest_forecast_interval = 3 #@param {type:"slider", min:1, max:24, step:1}
ecmwf_resolution = "0.25/0.25" #@param ["0.25/0.25", "0.5/0.5", "1.0/1.0"]
ecmwf_stream = "aifs-single" #@param ["aifs-single", "enfo", "wave"]

print(f"ECMWF Model Run Date: {ecmwf_model_run_date_str}")
print(f"ECMWF Model Cycle: {ecmwf_model_cycle}z")
print(f"ECMWF Max Forecast Hour: {ecmwf_max_forecast_hour}")
print(f"ECMWF Finest Forecast Interval: {ecmwf_finest_forecast_interval} hours")
print(f"ECMWF Resolution: {ecmwf_resolution}")
print(f"ECMWF Stream: {ecmwf_stream}")

# Define output directory for ECMWF JMV files
output_jmv_ecmwf_dir = '/content/jmv_output_ecmwf'
os.makedirs(output_jmv_ecmwf_dir, exist_ok=True)

# Initialize ECMWF OpenData client
client = Client()

# Generate list of forecast steps: 0, max_forecast_hour, and intermediate steps based on interval
forecast_steps = list(range(0, ecmwf_max_forecast_hour + 1, ecmwf_finest_forecast_interval))
if 0 not in forecast_steps:
    forecast_steps.insert(0, 0) # Ensure 0-hour forecast is always included

print(f"Generated forecast steps: {forecast_steps}")

# Extract relevant GRIB tags from PARAM_SPECS for ECMWF
# (PARAM_SPECS is defined in an earlier cell and should be globally available)
# The grib_tags are: '2t', '10u', '10v', 'msl', 'swh', 'mwd', 'mwp', 'gust'
ecmwf_grib_tags = [spec.grib_tag for spec in PARAM_SPECS.values()]
print(f"Parameters to retrieve: {ecmwf_grib_tags}")

# Map the user-selected ecmwf_stream to the actual API stream parameter
# 'aifs-single' maps to 'oper' for dates after 2026-05-12.
# 'wave' maps directly to 'wave'. 'enfo' maps directly to 'enfo'.
if ecmwf_stream == 'aifs-single':
    api_stream = 'oper'
elif ecmwf_stream == 'wave':
    api_stream = 'wave'
else:
    api_stream = ecmwf_stream

# Loop through forecast steps and parameters to download and convert
downloaded_files_count = 0
for step in forecast_steps:
    for param_grib_tag in ecmwf_grib_tags:
        # Construct target filename based on ECMWF conventions
        target_filename = f"ecmwf_{param_grib_tag}_{ecmwf_model_run_date_str.replace('-', '')}_{ecmwf_model_cycle}_f{step:03d}.grib2"
        local_grib_filepath = os.path.join('/tmp', target_filename)

        print(f"Attempting to download {param_grib_tag} for step {step}h...")
        try:
            client.retrieve(
                date=ecmwf_model_run_date_str,
                time=ecmwf_model_cycle,
                type='fc', # forecast
                step=step,
                param=param_grib_tag,
                stream=api_stream, # Use the mapped API stream ('oper', 'enfo', or 'wave')
                grid=ecmwf_resolution,
                format='grib',
                target=local_grib_filepath,
            )
            print(f"Successfully downloaded {os.path.basename(local_grib_filepath)}")

            print(f"Converting {os.path.basename(local_grib_filepath)} to JMV...")
            convert_grib_to_jmv(local_grib_filepath, output_jmv_ecmwf_dir)
            os.remove(local_grib_filepath) # Clean up GRIB file after conversion
            print(f"Conversion complete for {os.path.basename(local_grib_filepath)}")
            downloaded_files_count += 1

        except Exception as e:
            print(f"Error downloading or converting {param_grib_tag} for step {step}h: {e}")
            # Continue to the next parameter/step even if one fails

print(f"\nECMWF download and conversion attempt finished. Total files processed: {downloaded_files_count}")

# IMPORTANT: After running this cell, you will need to manually adjust the dropdown
# in cell `ccb5b56d` ('Select the folder containing your converted JMV files')
# to include '/content/jmv_output_ecmwf' as an option, or manually type it in.

ECMWF Model Run Date: 2026-07-09
ECMWF Model Cycle: 06z
ECMWF Max Forecast Hour: 240
ECMWF Finest Forecast Interval: 3 hours
ECMWF Resolution: 0.25/0.25
ECMWF Stream: aifs-single
Generated forecast steps: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186, 189, 192, 195, 198, 201, 204, 207, 210, 213, 216, 219, 222, 225, 228, 231, 234, 237, 240]
Parameters to retrieve: ['2t', 'gust', 'msl', '10u', '10v', 'mwd', 'mwp', 'swh']
Attempting to download 2t for step 0h...


20260709060000-0h-oper-fc.grib2:   0%|          | 0.00/633k [00:00<?, ?B/s]

Successfully downloaded ecmwf_2t_20260709_06_f000.grib2
Converting ecmwf_2t_20260709_06_f000.grib2 to JMV...
Conversion complete for ecmwf_2t_20260709_06_f000.grib2
Attempting to download gust for step 0h...
Error downloading or converting gust for step 0h: Cannot find index entries matching {'type': ['fc'], 'step': ['0'], 'param': ['gust']}
Attempting to download msl for step 0h...


20260709060000-0h-oper-fc.grib2:   0%|          | 0.00/496k [00:00<?, ?B/s]

Successfully downloaded ecmwf_msl_20260709_06_f000.grib2
Converting ecmwf_msl_20260709_06_f000.grib2 to JMV...
Conversion complete for ecmwf_msl_20260709_06_f000.grib2
Attempting to download 10u for step 0h...


...

Successfully downloaded ecmwf_msl_20260709_06_f144.grib2
Converting ecmwf_msl_20260709_06_f144.grib2 to JMV...
Conversion complete for ecmwf_msl_20260709_06_f144.grib2
Attempting to download 10u for step 144h...


20260709060000-144h-oper-fc.grib2:   0%|          | 0.00/850k [00:00<?, ?B/s]

Successfully downloaded ecmwf_10u_20260709_06_f144.grib2
Converting ecmwf_10u_20260709_06_f144.grib2 to JMV...
Conversion complete for ecmwf_10u_20260709_06_f144.grib2
Attempting to download 10v for step 144h...


20260709060000-144h-oper-fc.grib2:   0%|          | 0.00/843k [00:00<?, ?B/s]

Successfully downloaded ecmwf_10v_20260709_06_f144.grib2
Converting ecmwf_10v_20260709_06_f144.grib2 to JMV...
Conversion complete for ecmwf_10v_20260709_06_f144.grib2
Attempting to download mwd for step 144h...
Error downloading or converting mwd for step 144h: Cannot find index entries matching {'type': ['fc'], 'step': ['144'], 'param': ['mwd']}
Attempting to download mwp for step 144h...
Error downloading or converting mwp for step 144h: Cannot find index entries matching {'type': ['fc'], 'step': ['144'], 'param': ['mwp']}
Attempting to download swh for step 144h...
Error downloading or converting swh for step 144h: Cannot find index entries matching {'type': ['fc'], 'step': ['144'], 'param': ['swh']}
Attempting to download 2t for step 147h...
Error downloading or converting 2t for step 147h: 404 Client Error: Not Found for url: https://data.ecmwf.int/forecasts/20260709/06z/ifs/0p25/oper/20260709060000-147h-oper-fc.index
Attempting to download gust for step 147h...
Error downloading

### Method B: Upload Your Own GRIB2 Files

If you have your own GRIB2 files, you can upload them as a single `.zip` archive. The notebook will then extract and convert all GRIB2 files found within the archive.

In [ ]:
# Create a temporary directory for uploads
upload_dir = '/content/grib2_uploads'
os.makedirs(upload_dir, exist_ok=True)

print("Please upload a .zip file containing your GRIB2 files using the button below.")

uploaded = files.upload()

if uploaded:
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')
        # Save the uploaded file to the upload directory
        with open(os.path.join(upload_dir, fn), 'wb') as f:
            f.write(uploaded[fn])
    print("Upload complete. Proceed to the next cell to unzip and convert.")
else:
    print("No files were uploaded.")

Please upload a .zip file containing your GRIB2 files using the button below.


KeyboardInterrupt: 

In [15]:
# Define directories
upload_dir = '/content/grib2_uploads/'
output_jmv_local_dir = '/content/jmv_output_local/'
os.makedirs(output_jmv_local_dir, exist_ok=True)

# Find the uploaded zip file
zip_files = [f for f in os.listdir(upload_dir) if f.endswith('.zip')]

if not zip_files:
    print("No .zip file found in the upload directory. Please upload a .zip file first (Method B).")
elif len(zip_files) > 1:
    print("Multiple .zip files found. Processing the first one: {zip_files[0]}")
    zip_filepath = os.path.join(upload_dir, zip_files[0])
else:
    zip_filepath = os.path.join(upload_dir, zip_files[0])
    print(f"Found ZIP file: {os.path.basename(zip_filepath)}")

    # Unzip the file
    try:
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(upload_dir)
        print(f"Successfully unzipped {os.path.basename(zip_filepath)} into {upload_dir}")

        # Process extracted GRIB2 files
        grib2_files_found = 0
        for root, _, files_in_dir in os.walk(upload_dir):
            for file_name in files_in_dir:
                if file_name.endswith(('.grib2', '.grb2', '.grib')):
                    grib_filepath = os.path.join(root, file_name)
                    print(f"Converting {os.path.basename(grib_filepath)}...")
                    convert_grib_to_jmv(grib_filepath, output_jmv_local_dir)
                    grib2_files_found += 1

        if grib2_files_found == 0:
            print("No GRIB2 files found in the uploaded ZIP archive. Please ensure your ZIP contains files with .grib2, .grb2, or .grib extensions.")
        else:
            print(f"Conversion complete for {grib2_files_found} GRIB2 files.")

    except zipfile.BadZipFile:
        print(f"Error: {os.path.basename(zip_filepath)} is not a valid ZIP file.")
    except Exception as e:
        print(f"An error occurred during unzipping or conversion: {e}")

print("Local GRIB2 upload and conversion attempt finished.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/grib2_uploads/'

## 4. Package and Download Your JMV Files

The final step is to collect all the generated JMV files and package them into a single `.zip` archive. This archive can then be easily downloaded to your local computer.

In [16]:
#@markdown ### Select the folder containing your converted JMV files
#@markdown Choose the output directory based on which method (A or B) you used for conversion.
output_folder_to_package = '/content/jmv_output_ecmwf' #@param ["/content/jmv_output_ecmwf", "/content/jmv_output_local"]

print(f"Selected output folder for packaging: {output_folder_to_package}")

Selected output folder for packaging: /content/jmv_output_ecmwf


In [18]:
# Get model run date and cycle from global context (from cell grGuCW_g6mI_)
ecmwf_model_run_date_str = globals().get('ecmwf_model_run_date_str', 'UNKNOWN_DATE')
ecmwf_model_cycle = globals().get('ecmwf_model_cycle', '00')

# Format date and time for the filename
# Example: 2026-07-09 becomes 20260709
formatted_date = ecmwf_model_run_date_str.replace('-', '')
# Example: 06 becomes 0600
formatted_time = f"{ecmwf_model_cycle}00"

# Construct the new zip filename
zip_filename = f"Converted_JMV_Files_{formatted_date}-{formatted_time}"
zip_path = '/content/' + zip_filename # The zip file will be created in /content/

if os.path.isdir(output_folder_to_package) and len(os.listdir(output_folder_to_package)) > 0:
    try:
        # Create the zip archive
        shutil.make_archive(zip_path, 'zip', output_folder_to_package)
        print(f"Successfully created zip file: {zip_path}.zip")

        # Trigger download
        files.download(zip_path + '.zip')
        print("Download initiated. Check your browser's downloads.")

    except Exception as e:
        print(f"Error during zipping or downloading: {e}")
else:
    print(f"The selected folder '{output_folder_to_package}' does not exist or is empty. Please ensure you have successfully run one of the conversion methods (A or B) and that it generated JMV files.")

Successfully created zip file: /content/Converted_JMV_Files_20260709-0600.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Check your browser's downloads.
